In [1]:
import torch
import time
import numpy as np

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

import sys
from pathlib import Path
# sys.path.insert(0, str(Path.cwd() / "src"))

from model_alignment_lab.utils.helpers import generate_response, format_example
from model_alignment_lab.evaluation.eval import log_parser, evaluate_to_dataframe

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

C:\Users\DFS\Desktop\gitrepo\workshops\model-alignment-lab\venv_x86\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cpu')

In [6]:
## Path Setup
root = Path.cwd()
output_dir = root.parent/"outputs"
datasets_dir = root.parent/"datasets"/"structured_json"
root

WindowsPath('C:/Users/DFS/Desktop/gitrepo/workshops/model-alignment-lab/notebooks')

In [7]:
print(root.is_dir())
print(output_dir.is_dir())
print(datasets_dir.is_dir())

True
True
True


# 1. Base Model Preparation

In [2]:
MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float32
)

model = model.to(device)
model.eval()

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 290/290 [00:00<00:00, 743.08it/s]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 960, padding_idx=2)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=960, out_features=960, bias=False)
          (k_proj): Linear(in_features=960, out_features=320, bias=False)
          (v_proj): Linear(in_features=960, out_features=320, bias=False)
          (o_proj): Linear(in_features=960, out_features=960, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=960, out_features=2560, bias=False)
          (up_proj): Linear(in_features=960, out_features=2560, bias=False)
          (down_proj): Linear(in_features=2560, out_features=960, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((960,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((960,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((960,), eps=1e-05)
    (r

In [3]:
prompt = """Return ONLY valid JSON.

Available tools:
- search_flights(origin, destination, departure_date, return_date)
- search_hotels(location, check_in_date, check_out_date)
- plan_trip(destination, start_date, end_date, budget_level)
- check_weather(location, date)

Use exact tool names and exact argument keys.

User request:
I want to go from San Diego to New York from 2026-05-10 to 2026-05-15 and keep it a medium budget."""

response = generate_response(model, tokenizer, prompt)
print(response)

Sure, here are the tools you requested:

1. Search Flights:
- San Diego to New York:
- 2026-05-10 to 2026-05-15
- Medium Budget

2. Search Hotels:
- San Diego:
- 2026-05-10 to 2026-05-15
- Medium Budget

3. Plan Trip:
- San Diego:
- 2026-05-10 to 2026-05-15
- Medium Budget

4. Check Weather:
- San Diego:
- 2026-05-10 to 2026-05-15
- Medium Budget

5. Search for Accommod


In [11]:
result = generate_response(
    model,
    tokenizer,
    prompt,
    max_new_tokens=30,
    benchmark=True
)
# print(result["text"])
# print()
print(f"Total Time: {np.round(result['total_time_s'],2)}")
print(f"Generated Tokens: {np.round(result['generated_tokens'],2)}")
print(f"Tokens per second: {np.round(result['tokens_per_second'],2)}")

Total Time: 4.41
Generated Tokens: 30
Tokens per second: 6.8


In [12]:
train_path = datasets_dir.joinpath("TRAIN_travel_tool_routing_100_samples_prompted_v2.jsonl")
test_path = datasets_dir.joinpath("TEST_travel_tool_routing_20_samples_prompted_v2.jsonl")
val_path = datasets_dir.joinpath("VAL_travel_tool_routing_20_samples_prompted_v2.jsonl")

In [13]:
print(train_path.is_file())
print(test_path.is_file())
print(val_path.is_file())

True
True
True


In [14]:
from datasets import load_dataset

train_dataset = load_dataset("json",
                             data_files={
                                 "train": str(train_path)
                             }
                            )["train"]
test_dataset = load_dataset("json",
                            data_files={
                                "test": str(test_path)
                            }
                           )["test"]
val_dataset = load_dataset("json",
                           data_files={
                               "val":str(val_path)
                           }
                          )["val"]

In [15]:
train_dataset

Dataset({
    features: ['messages'],
    num_rows: 100
})

In [16]:
train_dataset["messages"][0]

[{'role': 'user',
  'content': 'Return ONLY valid JSON.\n\nAvailable tools:\n- search_flights(origin, destination, departure_date, return_date)\n- search_hotels(location, check_in_date, check_out_date)\n- plan_trip(destination, start_date, end_date, budget_level)\n- check_weather(location, date)\n\nUse exact tool names and exact argument keys.\n\nUser request:\nQuick question: Show me the weather for Paris on 2026-08-26. Please respond with JSON only.'},
 {'role': 'assistant',
  'content': '{"tool": "check_weather", "arguments": {"location": "Paris", "date": "2026-08-26"}}'}]

In [17]:
train_dataset = train_dataset.map(format_example, fn_kwargs={"tokenizer":tokenizer})
val_dataset = val_dataset.map(format_example, fn_kwargs={"tokenizer":tokenizer})
test_dataset = test_dataset.map(format_example, fn_kwargs={"tokenizer":tokenizer})

Map: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 360.07 examples/s]


In [18]:
train_dataset

Dataset({
    features: ['messages', 'text'],
    num_rows: 100
})

In [19]:
print(train_dataset["text"][0])

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Return ONLY valid JSON.

Available tools:
- search_flights(origin, destination, departure_date, return_date)
- search_hotels(location, check_in_date, check_out_date)
- plan_trip(destination, start_date, end_date, budget_level)
- check_weather(location, date)

Use exact tool names and exact argument keys.

User request:
Quick question: Show me the weather for Paris on 2026-08-26. Please respond with JSON only.<|im_end|>
<|im_start|>assistant
{"tool": "check_weather", "arguments": {"location": "Paris", "date": "2026-08-26"}}<|im_end|>



# 2. LoRA Finetune 

### LoRA Setup

In [20]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj"]
)


In [21]:
model = get_peft_model(model, peft_config).to(model.device)
model.train()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(49152, 960, padding_idx=2)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=960, out_features=960, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=960, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=960, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [22]:
model.print_trainable_parameters()

trainable params: 1,638,400 || all params: 363,459,520 || trainable%: 0.4508


## Training

In [23]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=str(output_dir/"smollm-tool-routing-lora"),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=False,
    dataset_text_field="text",
    use_cpu=False,
    bf16=False,
    fp16=False,
)

In [24]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

Tokenizing eval dataset: 100%|████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 417.53 examples/s]


In [ ]:
trainer.train()

In [ ]:
model

In [ ]:
final_path = str(output_dir/"smollm-tool-routing-lora")
trainer.model.save_pretrained(f"{final_path}/{ts}_final_adapter_v1")
tokenizer.save_pretrained(f"{final_path}/{ts}_final_adapter_v1")

## Model Evaluation

In [ ]:
trainer.state.log_history

In [ ]:
epoch,loss,eval_loss = log_parser(trainer)

print(epoch)
print(loss)
print(eval_loss)

In [ ]:
import matplotlib.pyplot as plt

figure, ax = plt.subplots()
ax.plot(epoch, loss, marker="o", label="Train Loss")
ax.plot(epoch, eval_loss, marker="x", label="Eval Loss")

ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training vs Evaluation Loss")

ax.legend()
ax.grid()

plt.show()

In [ ]:
prompt

In [ ]:
response = generate_response(model, tokenizer, prompt)
print(response)

In [ ]:
print(test_dataset["text"][0])

In [ ]:
df = evaluate_to_dataframe(model, tokenizer, test_dataset)
df.head()

In [ ]:
df.tool_match.value_counts()

In [ ]:
df.arguments_match.value_counts()

In [ ]:
df.valid_json.value_counts()

## Train mas

In [ ]:
# Increase number of epochs
trainer.args.num_train_epochs = 6
trainer.train()

## Model v2 Eval

In [ ]:
epoch,loss,eval_loss = log_parser(trainer)

print(epoch)
print(loss)
print(eval_loss)

In [ ]:
figure, ax = plt.subplots()
ax.plot(epoch, loss, marker="o", label="Train Loss")
ax.plot(epoch, eval_loss, marker="x", label="Eval Loss")

ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training vs Evaluation Loss")

ax.legend()
ax.grid()

plt.show()

In [ ]:
prompt 

In [ ]:
generate_response(model, tokenizer, prompt)

In [ ]:
df_2 = evaluate_to_dataframe(model, tokenizer, test_dataset)
df_2.head()

In [ ]:
df_2.tool_match.value_counts()

In [ ]:
df_2.arguments_match.value_counts()

In [ ]:
df_2.valid_json.value_counts()

In [ ]:
trainer.model.save_pretrained(f"{final_path}/{ts}_final_adapter_v2")
tokenizer.save_pretrained(f"{final_path}/{ts}_final_adapter_v2")